# GPU FEM benchmark on a Kaggle T4 (prepared, not run)

This notebook runs the periodic unit-cell FEM (`fem/`) on a Kaggle T4 GPU. It compares CPU PARDISO
with GPU cuDSS and `torch.sparse.spsolve`, and CPU with CUDA assembly.

It has not been run. The GPU results in this repository come from a local GeForce GTX 1650
(`results/gpu_gtx1650/`). The notebook also predates the `cudss32ir` solver (complex64
factorisation with complex128 iterative refinement), which is the fastest exact option measured.
Add it to `--solvers` before using the notebook.

Setup: attach a private Kaggle dataset that holds a zip of this repository
(`git archive -o fem-src.zip HEAD`). Accelerator: GPU T4. Internet: on (for pip).

In [ ]:
import os, shutil, subprocess, sys, glob
src = glob.glob('/kaggle/input/*/fem-src.zip')[0]
shutil.rmtree('/kaggle/working/fem', ignore_errors=True)
subprocess.run(['unzip', '-q', src, '-d', '/kaggle/working/fem'], check=True)
os.chdir('/kaggle/working/fem')
print(os.listdir('.'))

In [ ]:
%%bash
pip install -q ezdxf shapely pypardiso
pip install -q --index-url https://gmsh.info/python-packages-dev-nox gmsh
pip install -q "nvmath-python[cu12]" cupy-cuda12x
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
cd /kaggle/working/fem
python -m pytest -q tests 2>&1 | tail -3

In [ ]:
%%bash
cd /kaggle/working/fem
python scripts/bench.py --designs class5 class8 class9 class19 --devices cpu cuda --solvers pardiso cudss torch --out results/bench_kaggle_t4.json

In [ ]:
%%bash
cd /kaggle/working/fem
# one full GPU sweep for the report (cuDSS, 201 points)
python -m fem.sweep --design class9 --device cuda --solver cudss --tag gpu_cudss --quiet
cp -r results /kaggle/working/results_out